In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
def get_page_html(url: str) -> str:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1400")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)
        time.sleep(2)
        html = driver.page_source
    finally:
        driver.quit()

    return html

In [3]:
def extract_features(soup):
    data = {}

    rows = soup.select("#fiche_caracteristiques table tr")
    for row in rows:
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue

        key = th.get_text(" ", strip=True)
        value = td.get_text(" ", strip=True)

        data[key] = value

    return data

In [4]:
def extract_descriptions(soup):
    data = {}

    section = soup.select_one("#fiche_descriptions")
    if not section:
        return data

    current_side = None

    for tag in section.find_all(["h3", "p", "span", "a"]):
        if tag.name == "h3":
            current_side = tag.get_text(strip=True)

        elif current_side == "Obverse":
            if tag.name == "p" and "Script:" not in tag.text and "Lettering:" not in tag.text:
                data["Obverse Description"] = tag.get_text(" ", strip=True)

            if "Script:" in tag.text:
                data["Obverse Script"] = tag.get_text(" ", strip=True).replace("Script:", "").strip()

            if tag.get("id") == "obverse_lettering":
                data["Obverse Lettering"] = tag.get_text(" ", strip=True)

        elif current_side == "Reverse":
            if tag.name == "p" and "Script:" not in tag.text and "Lettering:" not in tag.text:
                data["Reverse Description"] = tag.get_text(" ", strip=True)

            if "Script:" in tag.text:
                data["Reverse Script"] = tag.get_text(" ", strip=True).replace("Script:", "").strip()

            if tag.get("id") == "reverse_lettering":
                data["Reverse Lettering"] = tag.get_text(" ", strip=True)

        elif current_side == "Printer" and tag.name == "a":
            data["Printer"] = tag.get_text(" ", strip=True)

    return data

In [5]:
def extract_images_and_credit(soup):
    data = {}

    images = soup.select("#fiche_photo a.coin_pic")
    if len(images) > 0:
        data["Image Front"] = images[0].get("href")
    if len(images) > 1:
        data["Image Back"] = images[1].get("href")

    credit = soup.select_one("#fiche_photo p.mentions")
    if credit:
        data["Image Credit"] = credit.get_text(" ", strip=True)

    return data

In [6]:
def extract_note_title(soup):
    title = soup.select_one("h1")
    if title:
        return title.get_text(" ", strip=True)
    return None

In [12]:
def scrape_note(note_url: str) -> pd.DataFrame:
    html = get_page_html(note_url)
    soup = BeautifulSoup(html, "html.parser")

    features = extract_features(soup)
    descriptions = extract_descriptions(soup)
    images = extract_images_and_credit(soup)
    title = extract_note_title(soup)

    data = {
        "Note Title": title,
        **features,
        **descriptions,
        **images,
        "Note URL": note_url
    }

    return data

In [8]:
# -------------------------
# MAIN
# -------------------------

# Load note links from previous step
df_links = pd.read_csv("numista_note_links.csv")

In [9]:
# TEMP FILTER - ONE NOTE FOR TESTING
df_links = df_links[df_links["note_url"] == "https://en.numista.com/247092"]

In [13]:
all_rows = []

for _, row in df_links.iterrows():
    print("Scraping:", row["note_url"])
    note_data = scrape_note(row["note_url"])
    all_rows.append(note_data)

df_notes = pd.DataFrame(all_rows)

print(df_notes.T)

df_notes.to_csv("numista_notes_detailed.csv", index=False)

Scraping: https://en.numista.com/247092
                                                                     0
Note Title                                                     5 Korun
Issuer                                                        Slovakia
Period                                          Republic ( 1939-1945 )
Type                                    Standard circulation banknotes
Year                                                              1945
Value                                                          5 Korún
Currency                                          Koruna ( 1939-1945 )
Composition                                                      Paper
Size                                                       122 × 59 mm
Shape                                                      Rectangular
Issued                                                15 February 1945
Demonetized                                            31 October 1945
Number                               